In [2]:
!nvidia-smi -L
!pip -q install "transformers>=4.40" accelerate
!rm -rf repo && git clone -q -b sadia https://github.com/abdulqudoos181005/AI-Legal-Helper-for-Pakistan.git repo
%cd repo
!ls data/processed

import csv, json, os, random, time, sys
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
sys.path.insert(0, os.getcwd())
from src.preprocess import normalize
from src.metrics import ranking_metrics, per_section_f1

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABELS = list(json.load(open("src/labels.json", encoding="utf-8")).keys())
L2I = {l: i for i, l in enumerate(LABELS)}
rows = list(csv.DictReader(open("data/processed/dataset.csv", encoding="utf-8")))
train = [r for r in rows if r["split"] == "train"]
val = [r for r in rows if r["split"] == "val"]
def Y(rs):
    m = np.zeros((len(rs), len(LABELS)), dtype=np.float32)
    for i, r in enumerate(rs):
        for s in json.loads(r["sections"]):
            m[i, L2I[s]] = 1
    return m
Ytr, Yva = Y(train), Y(val)
print(device, "train", len(train), "val", len(val))

GPU 0: Tesla T4 (UUID: GPU-5eedf83c-d054-3093-e83e-85d0a41c62f0)
/content/repo
dataset.csv  ilsi_clean.csv  synthetic.csv  synthetic_review_sample.csv
cuda train 1507 val 385


In [3]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(Dataset):
    def __init__(self, rs, y):
        self.enc = tok([normalize(r["text"]) for r in rs], truncation=True, max_length=MAX_LEN, padding="max_length")
        self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.y[i])
        return item

ds_tr, ds_va = DS(train, Ytr), DS(val, Yva)
pos = Ytr.sum(0); neg = len(Ytr) - pos
pos_weight = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1, 20), dtype=torch.float32, device=device)
subsets = {
    "synthetic_all": [i for i, r in enumerate(val) if r["source"] == "synthetic"],
    "synthetic_en": [i for i, r in enumerate(val) if r["source"] == "synthetic" and r["language"] == "en"],
    "synthetic_roman_ur": [i for i, r in enumerate(val) if r["source"] == "synthetic" and r["language"] == "roman_ur"],
    "ilsi": [i for i, r in enumerate(val) if r["source"] == "ilsi"],
}

@torch.no_grad()
def predict_logits(model, ds, bs=64):
    model.eval(); out = []
    for b in DataLoader(ds, batch_size=bs):
        b = {k: v.to(device) for k, v in b.items() if k != "labels"}
        out.append(model(**b).logits.float().cpu().numpy())
    return np.concatenate(out)

def evaluate(logits):
    p = 1 / (1 + np.exp(-logits))
    return {n: ranking_metrics(Yva[idx], p[idx]) for n, idx in subsets.items() if idx}

def train_run(lr, epochs=8, bs=16, patience=2):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), problem_type="multi_label_classification").to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    dl = DataLoader(ds_tr, batch_size=bs, shuffle=True)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * len(dl) * epochs), len(dl) * epochs)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best, best_state, bad, history = None, None, 0, []
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot = 0
        for b in dl:
            b = {k: v.to(device) for k, v in b.items()}
            labels = b.pop("labels")
            loss = loss_fn(model(**b).logits, labels)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad(); tot += loss.item()
        res = evaluate(predict_logits(model, ds_va))
        key = (res["synthetic_all"]["top3_recall"], res["synthetic_all"]["micro_f1"])
        history.append({"lr": lr, "epoch": ep, "train_loss": round(tot / len(dl), 4),
                        **{f"{n}_{m}": v for n, r in res.items() for m, v in r.items() if m != "samples"}})
        print(f"lr={lr} ep={ep} loss={tot/len(dl):.4f} syn top3={key[0]:.3f} microF1={key[1]:.3f} ({time.time()-t0:.0f}s)")
        if best is None or key > best:
            best, bad = key, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print("early stop"); break
    model.load_state_dict(best_state)
    return model, best, history

runs, history = {}, []
for lr in (2e-5, 5e-5):
    model, score, hist = train_run(lr)
    runs[lr] = (score, model); history += hist
best_lr = max(runs, key=lambda k: runs[k][0])
model = runs[best_lr][1]
print("best lr", best_lr, runs[best_lr][0])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

... (2 outputs left out)

lr=2e-05 ep=1 loss=0.9734 syn top3=0.165 microF1=0.092 (36s)
lr=2e-05 ep=2 loss=0.8663 syn top3=0.470 microF1=0.201 (33s)
lr=2e-05 ep=3 loss=0.7910 syn top3=0.609 microF1=0.318 (35s)
lr=2e-05 ep=4 loss=0.7312 syn top3=0.722 microF1=0.388 (36s)
lr=2e-05 ep=5 loss=0.6780 syn top3=0.791 microF1=0.396 (36s)
lr=2e-05 ep=6 loss=0.6326 syn top3=0.800 microF1=0.389 (36s)
lr=2e-05 ep=7 loss=0.5990 syn top3=0.809 microF1=0.389 (36s)
lr=2e-05 ep=8 loss=0.5842 syn top3=0.800 microF1=0.395 (36s)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lr=5e-05 ep=1 loss=0.9490 syn top3=0.191 microF1=0.090 (36s)
lr=5e-05 ep=2 loss=0.7912 syn top3=0.643 microF1=0.312 (36s)
lr=5e-05 ep=3 loss=0.6659 syn top3=0.748 microF1=0.381 (36s)
lr=5e-05 ep=4 loss=0.5485 syn top3=0.870 microF1=0.468 (36s)
lr=5e-05 ep=5 loss=0.4553 syn top3=0.922 microF1=0.509 (36s)
lr=5e-05 ep=6 loss=0.3806 syn top3=0.896 microF1=0.536 (36s)
lr=5e-05 ep=7 loss=0.3272 syn top3=0.887 microF1=0.553 (36s)
early stop
best lr 5e-05 (0.9217, 0.509)


In [4]:
os.makedirs("results", exist_ok=True)
logits = predict_logits(model, ds_va)
res = evaluate(logits)
with open("results/distilbert_val.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["metric", "language", "value"])
    for n, r in res.items():
        for m, v in r.items(): w.writerow([m, n, v])
keys = list(dict.fromkeys(k for h in history for k in h))
with open("results/distilbert_grid.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(history)
with open("results/distilbert_val_scores.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["id"] + LABELS)
    for r, row in zip(val, logits): w.writerow([r["id"]] + [round(float(x), 5) for x in row])
for n, r in res.items(): print(n, r)
syn = subsets["synthetic_all"]
ps = per_section_f1(Yva[syn], 1/(1+np.exp(-logits[syn])), LABELS)
print({k: v for k, v in ps.items() if v[1] > 0})

model.save_pretrained("distilbert"); tok.save_pretrained("distilbert")
json.dump({"labels": LABELS, "max_len": MAX_LEN, "base_model": MODEL_NAME, "best_lr": best_lr,
           "temperature": 1.0}, open("distilbert/track_a_config.json", "w"), indent=2)
!zip -qr distilbert.zip distilbert results/distilbert_val.csv results/distilbert_grid.csv results/distilbert_val_scores.csv
from google.colab import files
files.download("distilbert.zip")

synthetic_all {'samples': 115, 'top1_accuracy': 0.6522, 'top3_recall': 0.9217, 'micro_f1': 0.509, 'macro_f1': 0.516}
synthetic_en {'samples': 88, 'top1_accuracy': 0.7273, 'top3_recall': 0.9432, 'micro_f1': 0.5085, 'macro_f1': 0.5257}
synthetic_roman_ur {'samples': 27, 'top1_accuracy': 0.4074, 'top3_recall': 0.8519, 'micro_f1': 0.5106, 'macro_f1': 0.5421}
ilsi {'samples': 270, 'top1_accuracy': 0.563, 'top3_recall': 0.7889, 'micro_f1': 0.4329, 'macro_f1': 0.2631}
{'302': (0.5, 4), '319': (0.5714, 5), '324': (0.2222, 5), '337-A': (0.4545, 5), '337-F': (0.5217, 6), '339': (0.5, 3), '340': (0.2857, 4), '354': (0.5714, 4), '354A': (0.625, 5), '363': (0.7273, 4), '365': (0.5, 3), '365-A': (0.6667, 5), '379': (0.3077, 6), '380': (0.2609, 4), '382': (0.7143, 5), '384': (0.375, 4), '392': (0.4444, 4), '395': (0.5333, 4), '403': (0.7273, 5), '406': (0.2857, 4), '411': (0.4286, 3), '420': (0.4444, 3), '441': (0.7273, 5), '447': (0.6667, 5), '448': (0.5333, 4), '489-F': (0.9091, 5), '500': (0.6667,

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>